# NTS base case (REE España)

CasADi pipeline notebook. Mirrors `main.py`: build the DAE, run the steady-state + small-signal analysis (`ini`), then a `v_ref_4` step-change simulation (`run`).

Run this notebook with `nts/cases/base/` as the working directory so the relative paths to `nts_base.hjson` and `nts_base_xy_0.json` resolve.

## Imports

In [ ]:
import time

import numpy as np
from matplotlib import pyplot as plt

from pydae import ssa
from pydae.bps import BpsBuilder
from pydae.core.builder import CasadiBuilder
from pydae.core.model import CasadiModel
from pydae.bps.utils.reporter import report_all
from pydae.bps.utils.validator import validate_all
from pydae.bps.lines import change_line
from pydae.utils import read_data

DATA = 'nts_base.hjson'      # network description with buses, generators, lines and reference results
XY_0 = 'nts_base_xy_0.json'  # saved initial guess for the Newton-Raphson solver

## Build the model

Assemble the CasADi DAE from the HJSON description and fold it into an SX graph.

In [ ]:
def build():
    grid = BpsBuilder(DATA, use_casadi=True)
    grid.construct('nts_base')          # concatenate all component equations into grid.sys_dict
    return CasadiBuilder(grid.sys_dict).build()

## `ini()` — steady state and small-signal analysis

Apply the line 2–3 impedance adjustment, solve the load flow, print the report and validate against the reference values.

In [ ]:
model = CasadiModel(build())        # runtime model wrapping the CasADi graph
model.decimation = 10               # store every 10th integration step
change_line(model, {"bus_j": "2", "bus_k": "3",
                    "X_pu": 0.6, "R_pu": 0.0, "Bs_pu": 0.0, "S_mva": 100})

model.ini({}, XY_0)                 # Newton-Raphson load-flow initialization

report_all(model, DATA)
validate_all(model, DATA)

### Small-signal analysis

`ssa.damp` (eigenvalues, damping ratios, frequencies), `ssa.eig` (eigenvectors and participation factors), and `ssa.get_mode` (inter-area modes in the 0.1–0.5 Hz band).

In [ ]:
model.A_eval()
ssa.damp(model.A, model=model, sort='damp')
ssa.eig(model)
ssa.get_mode(model, f_min=0.1, f_max=0.5)

## `run()` — v_ref_4 step-change simulation

2 s of steady operation, then a +0.018 pu step on `v_ref_4` (voltage reference of generator 4), integrated for 40 s.

In [ ]:
model = CasadiModel(build())        # runtime model wrapping the CasADi graph
model.decimation = 10               # store every 10th integration step
change_line(model, {"bus_j": "2", "bus_k": "3",
                    "X_pu": 0.6, "R_pu": 0.0, "Bs_pu": 0.0, "S_mva": 100})

model.ini({}, XY_0)                 # load-flow initialization
model.run(2.0, {})                  # 2 s of steady operation before the disturbance

model.run(40.0, {"v_ref_4": model.get_value('v_ref_4') + 0.018})
model.post()                        # copy solver buffers into the public Time/X/Y/Z arrays

### Plot results vs. NTS reference

Generator 1 speed `omega_1` (top) and active power `p_g_1` in MW against the NTS reference curve (bottom). Saved to `nts_base.png`.

In [ ]:
nts_results = np.array(read_data(DATA)['results']['step_vref4']['data'])  # columns [t (s), p_g_1 (MW)]

fig, axes = plt.subplots(2, 1, figsize=(8, 8), sharex=True)

axes[0].plot(model.Time, model.get_values("omega_1"), label="omega_1", color="b")
axes[0].set_ylabel("Speed (pu)"); axes[0].legend(); axes[0].grid(True)

axes[1].plot(model.Time, model.get_values("p_g_1") * model.get_value('S_n_1') / 1e6,
             label="p_g_1", color="b")
axes[1].plot(nts_results[:, 0], nts_results[:, 1], label="p_g_1 (NTS ref)", color="r")
axes[1].set_ylabel("Power (MW)"); axes[1].legend(); axes[1].grid(True)
axes[1].set_ylim((1330, 1375))
axes[1].yaxis.set_major_locator(plt.MultipleLocator(5))
axes[1].set_xlabel("Time (s)")